## Extract Transform and Load (ETL) dos dados de movimentação de cargas do porto de Santos.
Disponibilizados pelo site da APS em: https://mensario.portodesantos.com.br/cargas/ \
O objetivo desse notebook é extrair a série temporal dos registros de movimentação de cargas contidos no arquivo .csv

Obs: Os dados estão em um formato "Long Table" precisaremos pivotar a agregar os dados para que fique em um período de registros mensais aos longo dos anos.

Referência: Python for Data Analysis, 3E (https://wesmckinney.com/book/)

In [1]:
#Importação da biblioteca necessária para esse procedimento:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/exportacao_cargas.csv", encoding = 'utf-8')

In [3]:
display(df)

,TOTAL_TEU,NUMERO_VIAGEM,TIPO_NAVEGACAO,MOVIMENTO,ANO,TERMINAIS,TOTAL_TONELADAS,NATUREZA_CARGA,TOTAL_UNID,MES,CLASSENAVIO,SENTIDO,TIPO_INSTALACAO,BERCOS,MERCADORIAS,ANO_MES
0,218,3948,CABOTAGEM,TRANSBORDO,2019,SANTOS BRASIL,"2338,58",CARGA CONTEINERIZADA,131,11,PORTA-CONTAINERS,EMBARQUE,PORTO ORGANIZADO,SBR 2,OUTRAS MERCADORIAS,2019-11-01
1,957,4162,LONGO CURSO,CONVENCIONAL,2019,BTP,12044,CARGA CONTEINERIZADA,585,11,PORTA-CONTAINERS,DESEMBARQUE,PORTO ORGANIZADO,BTP 03,OUTRAS MERCADORIAS,2019-11-01
2,2,4248,LONGO CURSO,REMOÇÃO,2019,DPWORLD (EMBRAPORT),"33,83",CARGA CONTEINERIZADA,1,11,PORTA-CONTAINERS,EMBARQUE,TUP,DPW 2,CARNES DIVERSAS,2019-11-01
3,2,3950,LONGO CURSO,CONVENCIONAL,2019,SANTOS BRASIL,"32,618",CARGA CONTEINERIZADA,1,11,PORTA-CONTAINERS,DESEMBARQUE,PORTO ORGANIZADO,SBR 2,CARNE DE AVES,2019-11-01
4,28,4215,CABOTAGEM,CONVENCIONAL,2019,SANTOS BRASIL,"460,73",CARGA CONTEINERIZADA,14,11,PORTA-CONTAINERS,EMBARQUE,PORTO ORGANIZADO,SBR 1,CARNE DE AVES,2019-11-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091729,0,1,CABOTAGEM,CONVENCIONAL,2013,OUTROS,33486,GRANEL LIQUIDO,0,4,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2013-04-01
1091730,0,1,CABOTAGEM,CONVENCIONAL,2013,OUTROS,32929,GRANEL LIQUIDO,0,5,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2013-05-01
1091731,0,1,CABOTAGEM,CONVENCIONAL,2013,OUTROS,30325,GRANEL LIQUIDO,0,6,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2013-06-01
1091732,0,1,CABOTAGEM,CONVENCIONAL,2014,OUTROS,30367,GRANEL LIQUIDO,0,12,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2014-12-01


In [4]:
#Filtrando somente as colunas necessárias para a formação da série temporal
filtrar_colunas = ['ANO', 'MES', 'TOTAL_TONELADAS', 'MERCADORIAS']
df_filtrado = df[filtrar_colunas].copy()

In [5]:
display(df_filtrado) #aqui teremos os registros de data, tipo de mercadoria e toneladas movimentadas

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS
0,2019,11,"2338,58",OUTRAS MERCADORIAS
1,2019,11,12044,OUTRAS MERCADORIAS
2,2019,11,"33,83",CARNES DIVERSAS
3,2019,11,"32,618",CARNE DE AVES
4,2019,11,"460,73",CARNE DE AVES
...,...,...,...,...
1091729,2013,4,33486,BUNKER (O.COMBUSTIVEL)
1091730,2013,5,32929,BUNKER (O.COMBUSTIVEL)
1091731,2013,6,30325,BUNKER (O.COMBUSTIVEL)
1091732,2014,12,30367,BUNKER (O.COMBUSTIVEL)


In [6]:
#Passando os valores de toneladas para tipo float e trocando separador de milhares ',' por '.'
df_filtrado['TOTAL_TONELADAS'] = pd.to_numeric(
    df_filtrado['TOTAL_TONELADAS'].str.replace(',', '.', regex=False),
    errors='coerce'
    )
print(df_filtrado['TOTAL_TONELADAS'].dtype)

#Verificando se ficaram valores Not a Number (NaN) 
nans = df_filtrado['TOTAL_TONELADAS'].isna().sum()
print(f"NaNs introduzidos na conversão: {nans}")

float64
NaNs introduzidos na conversão: 0


In [7]:
display(df_filtrado)

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS
0,2019,11,2338.580,OUTRAS MERCADORIAS
1,2019,11,12044.000,OUTRAS MERCADORIAS
2,2019,11,33.830,CARNES DIVERSAS
3,2019,11,32.618,CARNE DE AVES
4,2019,11,460.730,CARNE DE AVES
...,...,...,...,...
1091729,2013,4,33486.000,BUNKER (O.COMBUSTIVEL)
1091730,2013,5,32929.000,BUNKER (O.COMBUSTIVEL)
1091731,2013,6,30325.000,BUNKER (O.COMBUSTIVEL)
1091732,2014,12,30367.000,BUNKER (O.COMBUSTIVEL)


In [8]:
#Introduzindo nova coluna TOTAL_MENSAL que irá quantificar cada produto por mês e ano
#qualquer linha que tiver o mesmo par (mes/ano) também terá os mesmos valores em TOTAL_MENSAL
df_filtrado['TOTAL_MENSAL'] = df_filtrado.groupby(['ANO', 'MES'])['TOTAL_TONELADAS'].transform('sum')

# Verificação
display(df_filtrado)
display(df_filtrado.dtypes)

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS,TOTAL_MENSAL
0,2019,11,2338.580,OUTRAS MERCADORIAS,1.102905e+07
1,2019,11,12044.000,OUTRAS MERCADORIAS,1.102905e+07
2,2019,11,33.830,CARNES DIVERSAS,1.102905e+07
3,2019,11,32.618,CARNE DE AVES,1.102905e+07
4,2019,11,460.730,CARNE DE AVES,1.102905e+07
...,...,...,...,...,...
1091729,2013,4,33486.000,BUNKER (O.COMBUSTIVEL),9.779155e+06
1091730,2013,5,32929.000,BUNKER (O.COMBUSTIVEL),9.914380e+06
1091731,2013,6,30325.000,BUNKER (O.COMBUSTIVEL),9.324899e+06
1091732,2014,12,30367.000,BUNKER (O.COMBUSTIVEL),9.052196e+06


ANO                  int64
MES                  int64
TOTAL_TONELADAS    float64
MERCADORIAS         object
TOTAL_MENSAL       float64
dtype: object

In [9]:
#Verificação dos tipos de cargas na coluna MERCADORIAS
lista_cargas = df_filtrado['MERCADORIAS'].unique()
print(lista_cargas)

['OUTRAS MERCADORIAS' 'CARNES DIVERSAS' 'CARNE DE AVES'
 'COMBUSTÍVEIS ÓLEOS E PRODUTOS MINERAIS' 'PRODUTOS QUÍMICOS ORGÂNICOS'
 'PRODUTOS SIDERÚRGICOS' 'ÓLEO DE ORIGEM VEGETAL' 'TRIGO' 'CELULOSE'
 'CAFÉ' 'FERTILIZANTES NITROGENADOS' 'FARELO DE SOJA' 'SOLVENTES'
 'ENXOFRE' 'NAFTA' 'CARNE BOVINA' 'PRODUTOS QUÍMICOS INORGÂNICOS'
 'SEM CARGAS' 'PRODUTOS DIVERSOS DA INDÚSTRIA QUÍMICA'
 'FERTILIZANTES POTÁSSICOS'
 'VEÍCULOS AUTOMÓVEIS (ESPECIAIS, P TRANSP. DE CARGAS EM FÁBRICAS, ARMAZÉNS...)'
 'CARVÃO MINERAL' 'AÇÚCAR' 'METANOL' 'FERTILIZANTES'
 'FERTILIZANTES (MISTURAS)' 'PEIXES E OUTROS AQUÁTICOS' 'OUTROS VEÍCULOS'
 'SUCOS CÍTRICOS' 'SODA CÁUSTICA' 'ALGODÃO'
 'VEÍCULOS AUTOMÓVEIS (TRATORES)'
 'VEÍCULOS AUTOMÓVEIS (TRANSP. DE PASS. (<10 PASS.)+ VEIC. CORRIDA)'
 'ETANOL' 'SAL' 'VEÍCULOS AUTOMÓVEIS (CHASSIS COM MOTOR)' 'SOJA EM GRÃOS'
 'ACESSÓRIOS DE VEÍCULOS AUTOMÓVEIS' 'MOTOCICLETAS'
 'OUTROS COMBUSTÍVEIS ÓLEOS E PRODUTOS MINERAIS' 'MILHO' 'COQUE'
 'ANIMAIS VIVOS' 'CARNE SUÍNA' 'ÓLEO BRUTO

In [10]:
#Verificação dos tipos de carga com maior quantidade de movimentação, os 10 maiores:
ranking = (
    df_filtrado
    .groupby('MERCADORIAS')['TOTAL_TONELADAS']
    .sum()
    .sort_values(ascending=False)
)
print(ranking.head(10))

MERCADORIAS
OUTRAS MERCADORIAS             6.033445e+08
AÇÚCAR                         3.753183e+08
SOJA EM GRÃOS                  3.588089e+08
MILHO                          2.108808e+08
FARELO DE SOJA                 1.094906e+08
CELULOSE                       8.607687e+07
PRODUTOS QUÍMICOS ORGÂNICOS    6.927112e+07
ÓLEO DIESEL                    6.394272e+07
ÓLEO COMBUSTÍVEL               5.189361e+07
SUCOS CÍTRICOS                 4.492617e+07
Name: TOTAL_TONELADAS, dtype: float64


In [11]:
#Selecionaremos os 5 tipos com maior quantidade de movimentação
#Esses mesmos tipos de cargas são interessante de explorar em um série temporal devido a influência das safras nesses commodities
tipos = ['SOJA EM GRÃOS', 'OUTRAS MERCADORIAS', 'AÇÚCAR', 'FARELO DE SOJA', 'MILHO']
df_mercadorias = df_filtrado[df_filtrado['MERCADORIAS'].isin(tipos)].copy()
display(df_mercadorias) #observe a redução no número de linhas abaixo da visualização do data frame:

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS,TOTAL_MENSAL
0,2019,11,2338.580,OUTRAS MERCADORIAS,1.102905e+07
1,2019,11,12044.000,OUTRAS MERCADORIAS,1.102905e+07
5,2019,11,23.205,OUTRAS MERCADORIAS,1.102905e+07
7,2020,1,8670.000,OUTRAS MERCADORIAS,8.313562e+06
13,2019,12,227.000,OUTRAS MERCADORIAS,1.021121e+07
...,...,...,...,...,...
1090699,2016,1,1472.441,OUTRAS MERCADORIAS,7.832663e+06
1090702,2016,1,6555.580,OUTRAS MERCADORIAS,7.832663e+06
1090703,2015,11,2859.660,FARELO DE SOJA,9.843859e+06
1090704,2016,2,1322.840,OUTRAS MERCADORIAS,9.018650e+06


In [12]:
#Rotacionaremos o dataframe para obter um formato em linhas de cada mercadoria somada mes unicamente a cada ano
df_st = df_mercadorias.pivot_table(
    index=['ANO','MES'],
    columns='MERCADORIAS',
    values='TOTAL_TONELADAS',
    aggfunc='sum',
    fill_value=0 #aqui estamos tratando a ausência de um registro em qualquer período como 0 e não como dado faltante, então é um ponto de atenção necessário para qualquer replicação desse código,
)
df_st = df_st.sort_index(level=['ANO','MES']).reset_index()
df_st.columns.name = None

In [13]:
display(df_st)

,ANO,MES,AÇÚCAR,FARELO DE SOJA,MILHO,OUTRAS MERCADORIAS,SOJA EM GRÃOS
0,2005,1,581383.455,148641.973,67968.749,1675915.136,107966.285
1,2005,2,878015.791,207084.350,11623.584,1709254.262,390955.911
2,2005,3,610138.354,186525.305,12014.295,1730293.750,896454.461
3,2005,4,764282.986,340274.229,7687.881,1860562.606,858345.463
4,2005,5,1007703.226,330537.573,12814.725,2027806.818,1062494.325
...,...,...,...,...,...,...,...
252,2026,1,1561010.948,867668.554,1151919.295,2886627.208,699950.201
253,2026,2,1477114.383,625398.725,74968.470,2661940.849,3235398.809
254,2026,3,1159034.590,998819.742,4121.160,3166793.308,6095307.368
255,2026,4,953982.788,1062685.405,7957.515,2946290.344,6028032.753


In [14]:
#Juntaremos ao data frame de série temporal a variável porto com a movimentação total de todas as mercadorias 
porto = (
    df_filtrado
    .groupby(['ANO','MES'])['TOTAL_TONELADAS']
    .sum()
    .reset_index()
    .rename(columns={'TOTAL_TONELADAS': 'porto'})
)
df_st = df_st.merge(porto, on=['ANO','MES'], how='left')

In [15]:
df_st.rename(columns={'ANO':'ano','MES':'mes','AÇÚCAR':'acucar','FARELO DE SOJA':'farelo_soja','MILHO':'milho','OUTRAS MERCADORIAS':'outros','SOJA EM GRÃOS':'graos_soja'}, inplace=True)

In [16]:
display(df_st)

,ano,mes,acucar,farelo_soja,milho,outros,graos_soja,porto
0,2005,1,581383.455,148641.973,67968.749,1675915.136,107966.285,5.027936e+06
1,2005,2,878015.791,207084.350,11623.584,1709254.262,390955.911,5.316605e+06
2,2005,3,610138.354,186525.305,12014.295,1730293.750,896454.461,5.899909e+06
3,2005,4,764282.986,340274.229,7687.881,1860562.606,858345.463,5.850636e+06
4,2005,5,1007703.226,330537.573,12814.725,2027806.818,1062494.325,6.826361e+06
...,...,...,...,...,...,...,...,...
252,2026,1,1561010.948,867668.554,1151919.295,2886627.208,699950.201,1.272810e+07
253,2026,2,1477114.383,625398.725,74968.470,2661940.849,3235398.809,1.317256e+07
254,2026,3,1159034.590,998819.742,4121.160,3166793.308,6095307.368,1.688558e+07
255,2026,4,953982.788,1062685.405,7957.515,2946290.344,6028032.753,1.649898e+07


In [17]:
#Criaremos um index como tipo datetime
df_st['data'] = pd.to_datetime(dict(year=df_st['ano'], month=df_st['mes'], day=1))
df_st = df_st.set_index('data')
df_st.index = pd.DatetimeIndex(df_st.index, freq='MS')
df_st = df_st.drop(columns=['ano', 'mes'])
display(df_st.tail())

,acucar,farelo_soja,milho,outros,graos_soja,porto
data,,,,,,
2026-01-01,1561010.948,867668.554,1151919.295,2886627.208,699950.201,1.272810e+07
2026-02-01,1477114.383,625398.725,74968.470,2661940.849,3235398.809,1.317256e+07
2026-03-01,1159034.590,998819.742,4121.160,3166793.308,6095307.368,1.688558e+07
2026-04-01,953982.788,1062685.405,7957.515,2946290.344,6028032.753,1.649898e+07
2026-05-01,1751670.729,1095828.696,1531.100,3109297.291,5046449.644,1.636979e+07


In [18]:
df_st.to_csv('../data/processed/serietemporal.csv')